# 04 — The knobs, and two ideas that failed

Notebook 03 compared policies at one setting each. This one sweeps the
settings, which is where several of the project's actual conclusions live.

Four sweeps:

1. **The service window** — the headline tunable, and the rule it produced.
2. **The commitment window** — a design-review mitigation that measurement
   killed.
3. **Safety factor against execution noise** — the knob that exists for CARLA.
4. **Two dead ends** — the ride cap and the detour cap, swept until they
   admitted defeat.

The fourth is here on purpose. A sweep that shows an idea failing is a result,
and dropping those policies without showing why would make the story tidier
and less true.

Runtime: about five minutes at `"quick"`, half an hour at `"full"`.

In [ ]:
# --- Bootstrap ------------------------------------------------------------
# This notebook imports the project as `src.…`, which requires the repository
# root on the import path. `./jupyter-run.sh` puts it there; opening the file
# any other way does not. This cell finds the root by walking up from the
# working directory, so the notebook runs wherever it is launched from and
# whatever folder it sits in.
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
            if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Seeded runs only reproduce when Python's string hashing is pinned: several
# policies break ties with min() over a set, whose iteration order is
# randomised per process otherwise. The simulation refuses to run unpinned;
# this says so now rather than after a long cell.
if os.environ.get("PYTHONHASHSEED") in (None, "random"):
    print("WARNING: PYTHONHASHSEED is not pinned, so seeded runs will be "
          "refused.\n         Launch with ./jupyter-run.sh, or set "
          "PYTHONHASHSEED=0.")

print(f"repository root: {ROOT}")

In [ ]:
import os

# How much work this notebook does. Every notebook in this folder has this
# switch and honours the AUTOBUS_SCALE environment variable, so nothing here
# needs editing to run it bigger or smaller.
#
#   "quick"  the default: minutes, same conclusions, wider error bars
#   "full"   the sample sizes behind the figures in the submitted documents
#   "smoke"  a tiny end-to-end check that the notebook runs at all
#
# The outputs saved in this file were produced at "full".
SCALE = os.environ.get("AUTOBUS_SCALE", "quick")
print(f"running at SCALE = {SCALE!r}")

In [ ]:
import matplotlib.pyplot as plt
from cycler import cycler

SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

In [ ]:
# The campus map: the world behind every synthetic result in this project.
# stop_density=0.5 is load-bearing -- the generator's default of 0.15 gives a
# 6-stop map on this seed, on which every policy ties. See notebook 01.
CAMPUS = dict(n_seeds=17, bounds=(0.0, 0.0, 2000.0, 2000.0), seed=42,
              stop_density=0.5)

In [ ]:
import numpy as np
import pandas as pd

from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import compare_policies
from src.map.graph import PlanarStreetGraph

TRIALS = {"smoke": 2, "quick": 15, "full": 60}[SCALE]
REQUESTS = {"smoke": 40, "quick": 200, "full": 200}[SCALE]
PATIENCE = 900.0        # how long a passenger waits before giving up

BASE = dict(max_requests=REQUESTS, demand_interval_s=60.0, capacity=40,
            max_wait_s=PATIENCE)


def sweep(policies, **overrides):
    """
    Run a set of arms on identical seeds and tabulate the KPI means.
    :param policies: (label, factory) pairs.
    :param overrides: Trial settings replacing the defaults in BASE.
    :return: One row per arm.
    """
    settings = {**BASE, **overrides}
    results = compare_policies(
        policies, graph=PlanarStreetGraph.generate(**CAMPUS),
        n_trials=TRIALS, base_seed=0, progress=False, **settings)
    rows = []
    for label, trials in results.items():
        rows.append({
            "arm": label,
            "served %": 100 * np.mean([t.kpi.served_fraction for t in trials]),
            "wait s": np.mean([t.kpi.mean_wait_time for t in trials]),
            "p90 wait s": np.mean([t.kpi.p90_wait_time for t in trials]),
            "ride s": np.mean([t.kpi.mean_ride_time for t in trials]),
            "max ride s": np.mean([max(t.ride_times) if t.ride_times else 0
                                   for t in trials]),
        })
    return pd.DataFrame(rows).set_index("arm").round(1), results


print(f"{TRIALS} paired trials per arm, {REQUESTS} requests each")

## 1. The service window

The insertion policies promise every accepted passenger a pickup within a
**service window** of asking, and refuse anyone they cannot reach in time. A
refused request is not thrown away — it stays waiting and may be accepted by a
later plan.

The window is the single most important setting in the project, and the trade
is sharp in both directions:

- **Too tight** and the bus refuses passengers it could have served, chasing
  only the convenient ones.
- **Too loose** and the promise means nothing; the bus accepts people it will
  strand, and they abandon.

Passenger patience here is 900 s. Watch what happens at exactly that value.

In [ ]:
WINDOWS = [450.0, 600.0, 750.0, 900.0, 1200.0, 1800.0]

window_arms = []
for w in WINDOWS:
    window_arms.append((f"L0 W{w:.0f}",
                        lambda g, w=w: WindowInsertionPolicy(g, max_wait_s=w)))
    window_arms.append((f"L1 W{w:.0f}",
                        lambda g, w=w: ReorderingInsertionPolicy(g,
                                                                 max_wait_s=w)))

window_table, window_results = sweep(window_arms)
display(window_table)

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(13.5, 3.8))
for axis, column, title in zip(
        axes, ["served %", "wait s", "ride s"],
        ["Coverage", "Mean wait", "Mean ride"]):
    for colour, family in zip(SERIES, ["L0", "L1"]):
        values = [window_table.loc[f"{family} W{w:.0f}", column]
                  for w in WINDOWS]
        axis.plot(WINDOWS, values, marker="o", color=colour, label=family)
    axis.axvline(PATIENCE, ls="--", lw=1.1, color=INK_SOFT)
    axis.set_title(title)
    axis.set_xlabel("service window (s)")
    axis.grid(alpha=0.3)
axes[0].annotate("patience", (PATIENCE, axes[0].get_ylim()[0]), fontsize=8,
                 color=INK_SOFT, xytext=(4, 8), textcoords="offset points")
axes[0].legend()
figure.tight_layout()

### The rule: set the window to passenger patience

Coverage peaks at the window that equals patience, and it is not a
coincidence. Below it, the policy refuses passengers it could still have
reached in time. Above it, it accepts passengers who will give up before the
bus arrives — a promise it cannot keep.

Measured across 25 different maps, **W900 maximised coverage on 25 of 25**, so
this is a rule rather than an artefact of one map. Notebook 05 is where that
cross-map check is done.

Setting the window at or below patience also buys a real guarantee: **no
accepted request is ever abandoned.** The two values are not linked in code —
a sweep has to vary them deliberately, which is what this cell does.

### And the part that is easy to get backwards

The **worst** ride times occur at the **tightest** windows. A tight deadline
makes the bus chase pickups that are about to expire, deferring the dropoffs
of people already aboard. Widening the window relaxes that pressure and the
ride tail falls.

## 2. What repairing the order buys (L1 against L0)

The two families above differ by one thing: L1 takes its existing schedule,
pulls one request out at a time and re-inserts it wherever it now fits best,
keeping the move only if total passenger time strictly improves.

Its advantage is small on coverage and real on ride — and it **grows as the
window tightens**, because a tighter deadline constrains the schedule harder,
which is exactly when repairing its order pays.

In [ ]:
gap = pd.DataFrame({
    "window": WINDOWS,
    "ride L0": [window_table.loc[f"L0 W{w:.0f}", "ride s"] for w in WINDOWS],
    "ride L1": [window_table.loc[f"L1 W{w:.0f}", "ride s"] for w in WINDOWS],
})
gap["L1 - L0"] = (gap["ride L1"] - gap["ride L0"]).round(1)
display(gap.set_index("window"))

figure, axis = plt.subplots(figsize=(7.2, 3.6))
axis.bar([str(int(w)) for w in WINDOWS], gap["L1 - L0"], color=SERIES[0])
axis.axhline(0, color=INK_SOFT, lw=1)
axis.set_xlabel("service window (s)")
axis.set_ylabel("mean ride difference (s)")
axis.set_title("Reordering pays most when the window is tight (negative = L1 better)")
axis.grid(alpha=0.3, axis="y")
figure.tight_layout()

At `"full"` scale over 25 maps this comes to **-111.5 s of
ride, p ~ 2e-138**, rising from -19 s at W1800 to -229 s at W450. At `"quick"`
scale on one map, expect the direction rather than the magnitude — 15 trials
is not enough to resolve a 70 s difference, which is itself a useful lesson
about how much sampling these questions need.

**Why the ladder stops here.** L1 is one relocation at a time. The next rung
up (L2, "ruin and recreate") ejects a whole related subset and rebuilds it.
It was deliberately not built: its edge is fitting *more* requests, and
refusals barely moved under L1; it is randomised, which fights the seeded
reproducibility everything here depends on; and L1 is already **exhausted at
one pass** — a second sweep accepts 0.086% of its attempts.

## 3. The commitment window: a mitigation that measurement killed

The design review proposed a *committed segment*: the bus locks the first few
stops of its plan so re-planning cannot oscillate between destinations.

It sounds obviously right. Measured, it is **monotonically harmful**, and the
default changed from 2 to 0.

In [ ]:
COMMITMENTS = [0, 1, 2, 4, 8]

rows = []
for commitment in COMMITMENTS:
    table, _ = sweep(
        [(f"c={commitment}",
          lambda g: ReorderingInsertionPolicy(g, max_wait_s=PATIENCE))],
        commitment=commitment)
    rows.append(table.assign(commitment=commitment))

commitment_table = pd.concat(rows).set_index("commitment")
display(commitment_table[["served %", "wait s", "ride s"]])

**Why locking stops is harmful.** The policy plans from a
*projected* state — where the bus will be once the locked stops are done — but
a passenger's deadline is measured from when they asked. Lock enough stops and
the projection outruns the window: every new request arrives already expired,
and acceptance collapses.

The transferable rule is not "commitment 2 is bad", it is
**`commitment x median_leg` must stay well under the service window**. The
breakdown point tracks that ratio rather than sitting at a fixed number of
stops.

It is safe to lock nothing because the bus only ever re-plans when it is
**stationary at a stop**. It never abandons a leg it is driving. If re-planning
ever moved mid-leg — which is what CARLA would require — at least one committed
stop becomes physically necessary again.

## 4. When the driving does not match the plan

Everything above assumes the bus arrives exactly when the model said it would,
because on the graph the policy and the world consult the same travel times.
CARLA breaks that: traffic and signals make a leg take a time nobody predicted,
so a promise certified feasible can still be broken on the road.

`travel_noise` reproduces that failure on the graph by giving the *backend*
a noisier travel model than the policy plans against. `safety_factor` is the
answer to it: the policy inflates its own travel estimates, so a schedule it
certifies has slack in it.

A **breach** below is a fulfilled passenger whose actual wait exceeded the
window they were promised.

In [ ]:
NOISES = [0.0, 0.1, 0.2, 0.3]
SAFETY = [1.0, 1.25]
WINDOW = 750.0      # tighter than patience, or breaches are impossible to see

rows = []
for sigma in NOISES:
    for safety in SAFETY:
        table, results = sweep(
            [(f"sigma={sigma} k={safety}",
              lambda g, s=safety: ReorderingInsertionPolicy(
                  g, max_wait_s=WINDOW, safety_factor=s))],
            travel_noise=sigma, noise_seed=7)
        waits = [w for trials in results.values() for t in trials
                 for w in t.wait_times]
        rows.append({
            "noise sigma": sigma,
            "safety_factor": safety,
            "breach %": round(100 * np.mean([w > WINDOW for w in waits]), 2),
            "served %": round(table["served %"].iloc[0], 1),
            "ride s": round(table["ride s"].iloc[0], 1),
        })

noise_table = pd.DataFrame(rows).pivot(
    index="noise sigma", columns="safety_factor")
display(noise_table)

Raising the safety factor cuts the breach rate at no cost in
coverage — and pays for it in ride time, because a bus that assumes it is
slower than it is leaves gaps in its schedule. Since ride time is a graded KPI
and breach rate is not, **the default stays 1.0** and the knob is left for
CARLA, where the mismatch is real rather than simulated.

Note also the breaches at `sigma = 0`. Those are not model error: when nothing
at all can be scheduled, the policy drives to the nearest waiting passenger
anyway, deliberately ignoring the window. A late pickup beats a stationary
bus.

## 5. Two dead ends, swept until they admitted it

Both of these attacked the same problem — greedy has great wait times and a
terrible ride tail — from opposite directions. Both failed, and the *way* they
failed is the interesting part.

### `RideBoundedGreedyPolicy`: bound the ride from inside greedy

When any onboard passenger would be pushed past `ride_factor x` their direct
travel time, pickups are switched off until somebody is delivered. It never
rejects anyone.

In [ ]:
from src.algorithm.NearestUnservedPolicy import NearestUnservedPolicy
from src.algorithm.RideBoundedGreedyPolicy import RideBoundedGreedyPolicy
from src.algorithm.SkipStopFixedRoutePolicy import SkipStopFixedRoutePolicy

RIDE_FACTORS = [1.5, 2.0, 2.5, 3.0]
ride_table, _ = sweep(
    [(f"rf={f}", lambda g, f=f: RideBoundedGreedyPolicy(g, ride_factor=f))
     for f in RIDE_FACTORS]
    + [("Nearest (rf=inf)", lambda g: NearestUnservedPolicy(g))]
    + [("Skip-stop", lambda g: SkipStopFixedRoutePolicy(g))])
display(ride_table)

**The frontier is a straight diagonal back to plain greedy.**
Tightening the ride factor buys ride time and pays for it in coverage, one for
one, and it never arrives anywhere new. The decisive comparison is against
skip-stop *at equal coverage*: the fixed cycle delivers a bounded ride
structurally, for free, and the greedy policy cannot buy the same bound at any
setting.

The mechanism is worth stating, because it surprised us: the gate never
rejects anyone, but **deferring a pickup under a patience deadline is a soft
rejection** — the deferred passenger gives up. Any ride protection that delays
pickups costs coverage.

### `DeviatedFixedRoutePolicy`: add coverage to the fixed cycle

The mirror image. Keep the skip-stop backbone, but before driving to the next
scheduled stop, divert to nearby demand if the detour is small enough.

In [ ]:
from src.algorithm.DeviatedFixedRoutePolicy import DeviatedFixedRoutePolicy

DETOUR_FACTORS = [1.0, 1.25, 1.5, 2.0]
detour_table, _ = sweep(
    [(f"df={f}", lambda g, f=f: DeviatedFixedRoutePolicy(g, detour_factor=f))
     for f in DETOUR_FACTORS]
    + [("Skip-stop", lambda g: SkipStopFixedRoutePolicy(g)),
       ("Nearest", lambda g: NearestUnservedPolicy(g))])
display(detour_table)

`detour_factor = 1.0` is exactly skip-stop by construction,
which makes this a properly nested comparison. And then the cliff: one step
above 1.0 and the ride tail collapses onto plain greedy.

**The reason is a bound that was asserted and turned out to be false.** The
detour cap is measured against the cost of the hop to the *next* scheduled
stop, which can itself be large, and deviations chain across the whole demand
set. When scheduled stops are far apart, the policy permits large repeated
detours — which is greedy with extra steps. The claimed guarantee "ride is at
most `detour_factor x` one cycle" does not hold, and the policy's docstring
was corrected to retract it.

### What both failures taught

Each of these bolted a constraint onto a **fixed visit order** — greedy's, or
the cycle's. Neither could win, because the order itself was the problem. The
insertion policies do not constrain an order, they **choose** one, which is why
they reach parts of the frontier that neither of these could.

## Summary

| Knob | Answer | Confidence |
|---|---|---|
| Service window | **Set it to passenger patience** | Best coverage on 25/25 maps |
| Reordering (L1 vs L0) | **Worth it**, most at tight windows | p ~ 2e-138 pooled over 25 maps |
| Reordering passes | **One**; a second accepts 0.086% | Measured, 300 trials |
| Commitment window | **Zero**; locking stops is monotonically harmful | +2.0 pp served, p = 0.002 |
| `safety_factor` | **1.0**; it trades ride for a non-graded KPI | 3,840 runs over three maps |
| `ride_factor` | Abandoned — a diagonal back to greedy | — |
| `detour_factor` | Abandoned — a cliff onto greedy | — |

Next: **05 — the campaign**, which takes the settled configuration and asks
whether its result is statistically real and whether it survives on maps other
than this one.